
# 311 Complaints Near CUNY Campuses – Spring 2025 Final Project

## Introduction  
New York City’s 311 service allows the public to report non-emergency issues (like noise, parking, or heating complaints) across the city. In this project, we mash up two NYC open datasets – 311 Service Requests (2010–Present) and CUNY University Campus Locations – to investigate our research question: What are the most common 311 complaints reported within a 0.5-mile radius of CUNY campuses? By combining these data, we aim to identify prevalent community issues around campus neighborhoods. This integration demonstrates several key data science techniques (using APIs, data wrangling with pandas DataFrames, writing custom functions for geolocation filtering, and visualizing results) in a single workflow. Data Sources: We use two public datasets: (1) the NYC 311 Service Requests dataset, which contains all 311 complaints citywide since 2010
motherduck.com
; and (2) the City University of New York Campus Locations dataset, which provides the geographic coordinates of all CUNY campuses
catalog.data.gov
. Both datasets are accessed via NYC/Open Data APIs. We will fetch the data, clean and prepare it, then perform spatial filtering to find complaints near each campus. Finally, we’ll analyze the filtered data to determine the top complaint types and visualize them with a bar chart.

**Concepts Demonstrated:**  
- Data access via API  
- DataFrames and wrangling with pandas  
- Functions & conditionals  
- Iteration  
- Geolocation filtering  
- Data visualization


In [2]:

import pandas as pd
import requests

url_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.json?$select=created_date,complaint_type,descriptor,borough,latitude,longitude&$limit=50000"
url_cuny = "https://data.ny.gov/resource/irqs-74ez.json?$select=campus,latitude,longitude"

response_311 = requests.get(url_311)
complaints_data = response_311.json()

response_cuny = requests.get(url_cuny)
campus_data = response_cuny.json()

df311 = pd.DataFrame(complaints_data)
df_cuny = pd.DataFrame(campus_data)


In [12]:

df311['latitude'] = pd.to_numeric(df311['latitude'], errors='coerce')
df311['longitude'] = pd.to_numeric(df311['longitude'], errors='coerce')
df311 = df311.dropna(subset=['latitude', 'longitude'])
df311['created_date'] = pd.to_datetime(df311['created_date'], errors='coerce')

df_cuny['latitude'] = pd.to_numeric(df_cuny['latitude'], errors='coerce')
df_cuny['longitude'] = pd.to_numeric(df_cuny['longitude'], errors='coerce')
df_cuny = df_cuny.dropna(subset=['latitude', 'longitude'])
df_cuny = df_cuny.drop_duplicates(subset=['campus'])


KeyError: 'latitude'

In [5]:

import math

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 3958.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c


In [15]:
# Check and standardize coordinate columns for df311
if 'latitude' in df311.columns and 'longitude' in df311.columns:
    df311['latitude'] = pd.to_numeric(df311['latitude'], errors='coerce')
    df311['longitude'] = pd.to_numeric(df311['longitude'], errors='coerce')
elif 'Latitude' in df311.columns and 'Longitude' in df311.columns:
    df311['latitude'] = pd.to_numeric(df311['Latitude'], errors='coerce')
    df311['longitude'] = pd.to_numeric(df311['Longitude'], errors='coerce')
elif 'location' in df311.columns:
    df311['latitude'] = df311['location'].apply(lambda x: float(x['latitude']) if isinstance(x, dict) and 'latitude' in x else None)
    df311['longitude'] = df311['location'].apply(lambda x: float(x['longitude']) if isinstance(x, dict) and 'longitude' in x else None)
else:
    raise KeyError("No valid latitude/longitude columns found in df311")

# Drop rows with missing or invalid coordinates
df311 = df311.dropna(subset=['latitude', 'longitude'])
df311['created_date'] = pd.to_datetime(df311['created_date'], errors='coerce')

# Standardize CUNY campus coordinates
if 'latitude' in df_cuny.columns and 'longitude' in df_cuny.columns:
    pass  # all good
elif 'Latitude' in df_cuny.columns and 'Longitude' in df_cuny.columns:
    df_cuny['latitude'] = pd.to_numeric(df_cuny['Latitude'], errors='coerce')
    df_cuny['longitude'] = pd.to_numeric(df_cuny['Longitude'], errors='coerce')
else:
    raise KeyError("No valid latitude/longitude columns found in df_cuny")

# Drop rows with missing campus coordinates and duplicates
df_cuny = df_cuny.dropna(subset=['latitude', 'longitude'])
df_cuny = df_cuny.drop_duplicates(subset=['campus'])


<ipython-input-15-c6ad2a722f98>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df311['created_date'] = pd.to_datetime(df311['created_date'], errors='coerce')


KeyError: 'No valid latitude/longitude columns found in df_cuny'

In [16]:

radius_miles = 0.5
df_near_campus = pd.DataFrame(columns=df311.columns)

for idx, campus in df_cuny.iterrows():
    c_lat = campus['latitude']
    c_lon = campus['longitude']
    distances = df311.apply(lambda row: haversine_distance(c_lat, c_lon, row['latitude'], row['longitude']), axis=1)
    close_complaints = df311[distances <= radius_miles].copy()
    close_complaints['campus'] = campus['campus']
    df_near_campus = pd.concat([df_near_campus, close_complaints], ignore_index=True)

df_near_campus = df_near_campus.drop_duplicates(subset=df311.columns.tolist())


KeyError: 'latitude'

In [10]:

complaint_counts = df_near_campus['complaint_type'].value_counts().reset_index()
complaint_counts.columns = ['Complaint Type', 'Count']
top10 = complaint_counts.head(10)
top10


,Complaint Type,Count


In [9]:

import matplotlib.pyplot as plt

top5 = complaint_counts.head(5)
plt.figure(figsize=(10,6))
plt.barh(top5['Complaint Type'], top5['Count'], color='skyblue')
plt.xlabel("Number of Complaints")
plt.title("Top 5 311 Complaint Types Near CUNY Campuses")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


NameError: name 'complaint_counts' is not defined


## Conclusion  
 By mashing up the 311 Service Requests data with CUNY campus locations, we found that noise complaints dominate the 311 issues around CUNY campuses, with residential noise being the top complaint type. Parking-related complaints (illegal parking and blocked driveways) are also prevalent near many campuses, indicating potential traffic and congestion challenges. Other issues like heating problems appear but to a lesser extent in comparison to noise and parking. These results suggest that students and residents around campuses most commonly deal with noise disturbances and parking concerns. Reflection on the Process: In this project, we successfully integrated two open datasets to answer a real-world question. We demonstrated data retrieval via an API, cleaning and preparing data using pandas, and performing a spatial filter using a custom function for distance calculations (leveraging our knowledge of functions, loops, and conditionals in Python). We then used DataFrame operations to aggregate the data and identify patterns, and finally visualized the results using a bar chart. Each step – from data wrangling to geolocation filtering to visualization – showcased important programming concepts from the course. By structuring the notebook as a self-contained report, we’ve ensured that anyone can follow the analysis without outside explanation. We clearly stated the objective, documented the methodology, and provided interpretation of the results. This approach and the findings not only fulfill the project requirements but also illustrate how open data can be combined to generate insights into urban issues affecting community spaces like CUNY campuses.
